# Song Lyrics Generator — Character-Level nano-GPT

This notebook adapts our Shakespearean nano-GPT to generate song lyrics. It turns out that lyrics are a *fantastic* corpus for a small transformer: they're highly structured (verses, choruses, line breaks, rhyme), consistent in style within a genre, and short enough per "document" that the model sees thousands of examples even from a modest dataset.

**Getting lyrics data:** This notebook expects a plain `.txt` file with lyrics separated by a delimiter (see the Data section). The [Kaggle 55k song lyrics dataset](https://www.kaggle.com/datasets/nikhilnayak123/5-million-song-lyrics-dataset) works well, as does any collection you assemble yourself from a genre you like. 

Make sure your Colab runtime is set to GPU (Runtime → Change runtime type → GPU).


In [ ]:
import os
import time
import math
import re
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- Hyperparameters ---
BATCH_SIZE       = 64
BLOCK_SIZE       = 256   # Context length in characters — lyrics have short lines, so 256 is plenty
MAX_ITERS        = 5000
EVAL_INTERVAL    = 250
LEARNING_RATE    = 3e-4
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
N_EMBD           = 384
N_HEAD           = 6
N_LAYER          = 6
DROPOUT          = 0.2
BETA1, BETA2     = 0.9, 0.95
EARLY_STOPPING_PATIENCE  = 8   # A little more generous than Shakespeare — lyrics loss can plateau briefly
EVAL_ITERS_FOR_LOSS      = 100

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f"Using device: {DEVICE}")


## 1. Data Loading and Tokenization

Lyrics files typically look like one song per block, separated by a blank line or a delimiter like `---`. We'll load the whole thing as a single string and let the character-level model figure out the structure — newlines, repeated chorus lines, rhyme patterns and all.  I have made two sample lyrics files, one with rap and another with country.


In [ ]:
LYRICS_FILE = 'lyrics_50k.txt'

if os.path.exists(LYRICS_FILE):
    print(f"Loading lyrics from '{LYRICS_FILE}'...")
    with open(LYRICS_FILE, 'r', encoding='utf-8', errors='replace') as f:
        raw = f.read()
    print(f"Loaded {len(raw):,} characters.")
else:
    print(f"'{LYRICS_FILE}' not found — using the built-in demo corpus.")
    print("For real results, point LYRICS_FILE at a lyrics dataset.")

# --- Light cleaning ---
# Keep newlines (they carry structure!), lowercase everything, and strip non-printable characters
def clean_lyrics(text):
    text = text.lower()
    # Keep letters, digits, common punctuation, spaces, and newlines since we want them in our generated lyrics
    text = re.sub(r"[^\x20-\x7e\n]", " ", text)
    # Collapse runs of blank lines to a single separator
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

data = clean_lyrics(raw)
print(f"After cleaning: {len(data):,} characters")
print(f"\nSample (first 300 chars):\n{data[:300]}")

In [ ]:
# --- Character-level tokenizer ---
chars     = sorted(set(data))
vocab_size = len(chars)
print(f"Vocabulary: {vocab_size} unique characters")
print("Characters:", repr(''.join(chars)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join(itos[i] for i in l)

# Quick sanity check
sample = "carry me home"
assert decode(encode(sample)) == sample, "Encode/decode mismatch!"
print("Tokenizer check passed!")


In [ ]:
# --- Train / val split and data loader ---
n          = len(data)
train_data = torch.tensor(encode(data[:int(n * 0.9)]), dtype=torch.long)
val_data   = torch.tensor(encode(data[int(n * 0.9):]),  dtype=torch.long)
print(f"Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}")

def get_batch(split):
    src       = train_data if split == 'train' else val_data
    ix        = torch.randint(len(src) - BLOCK_SIZE, (BATCH_SIZE,))
    x         = torch.stack([src[i:i+BLOCK_SIZE]   for i in ix])
    y         = torch.stack([src[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

xb, yb = get_batch('train')
print("Batch shapes:", xb.shape, yb.shape)
print("Sample context:", repr(decode(xb[0][:60].tolist())))


## 2. Model Definition

Exactly the same transformer architecture as the Shakespeare notebook — `CausalSelfAttention`, `MLP`, `Block`, `GPT`. The only thing that changes automatically is `vocab_size`, since lyrics have a slightly different character set than Shakespeare.

One small addition: we swap the manual attention computation for `F.scaled_dot_product_attention` (Flash Attention), which is faster and more memory-efficient on modern GPUs. This frees up headroom to use the same `N_LAYER=6` even with the lyrics vocabulary.


In [ ]:
# --- 2. Model Definition ---

@dataclass
class GPTConfig:
    block_size: int   = BLOCK_SIZE
    vocab_size: int   = vocab_size
    n_layer:    int   = N_LAYER
    n_head:     int   = N_HEAD
    n_embd:     int   = N_EMBD
    dropout:    float = DROPOUT
    bias:       bool  = True

class LayerNorm(nn.Module):
    """LayerNorm with optional bias (same as Shakespeare version)."""
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias   = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn       = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj       = nn.Linear(config.n_embd, config.n_embd,     bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head       = config.n_head
        self.n_embd       = config.n_embd
        self.dropout      = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # Reshape into (B, n_head, T, head_size)
        def reshape(t):
            return t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)

        # Flash Attention — faster than the manual version and no causal mask buffer needed
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    """One transformer block: self-attention then MLP, both with residual connections."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: the input embedding and output projection share parameters.
        # This cuts ~25% of parameters and tends to improve performance.
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        # GPT-2 style scaled init for residual projections
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Number of trainable parameters: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos     = torch.arange(T, dtype=torch.long, device=idx.device)
        x       = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss   = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Autoregressive generation from a prompt.
        - temperature: lower (e.g. 0.7) = more focused/predictable; higher = more surprising
        - top_k: if set, only sample from the top-k most likely characters at each step
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx


In [ ]:
# --- Instantiate and inspect the model ---
model_config = GPTConfig(vocab_size=vocab_size)
model        = GPT(model_config).to(DEVICE)

# Optional: compile for a speedup on PyTorch >= 2.0
# model = torch.compile(model)

xb, yb = get_batch('train')
logits, loss = model(xb, yb)
print(f"Initial loss: {loss.item():.4f}  (expected ~{math.log(vocab_size):.4f} for a random model)")


## 3. Training

Same training loop as Shakespeare, with two upgrades:

1. **Cosine LR schedule with warmup** — the learning rate ramps up over 100 steps, then decays smoothly to `min_lr`. This is standard for transformers and usually shaves a few points off the final loss compared to a fixed rate.
2. **Gradient clipping** — caps gradient norms at 1.0 to prevent the occasional large update from destabilizing training. Especially useful with lyrics data, which can have unusual token distributions (lots of repeated lines).


In [ ]:
# --- Evaluation helper ---
@torch.no_grad()
def estimate_loss(m):
    m.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS_FOR_LOSS)
        for k in range(EVAL_ITERS_FOR_LOSS):
            X, Y = get_batch(split)
            _, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

# --- Cosine LR schedule with linear warmup ---
def get_lr(it):
    warmup = 100
    min_lr = LEARNING_RATE / 10
    if it < warmup:
        return LEARNING_RATE * it / warmup
    if it > MAX_ITERS:
        return min_lr
    ratio = (it - warmup) / (MAX_ITERS - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (LEARNING_RATE - min_lr)


In [ ]:
# --- 3. Training Loop ---
print(f"Starting training on {DEVICE}...")

optimizer         = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))
best_val_loss     = float('inf')
epochs_no_improve = 0
train_losses, val_losses, iter_hist = [], [], []
t0 = time.time()

for it in tqdm(range(MAX_ITERS), desc="Training"):

    # Update learning rate
    lr = get_lr(it)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # Periodic evaluation
    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
        losses = estimate_loss(model)
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        iter_hist.append(it)
        tqdm.write(f"Step {it:5d} | lr {lr:.2e} | train {losses['train']:.4f} | val {losses['val']:.4f}")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_lyrics_model.pth')
            tqdm.write(f"  -> New best val loss {best_val_loss:.4f} — checkpoint saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                tqdm.write(f"Early stopping at step {it}.")
                break

    # Forward / backward
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # Gradient clipping
    optimizer.step()

print(f"\nTraining finished in {(time.time()-t0)/60:.1f} min")
print(f"Best validation loss: {best_val_loss:.4f}")

if os.path.exists('best_lyrics_model.pth'):
    model.load_state_dict(torch.load('best_lyrics_model.pth', map_location=DEVICE))
    print("Best checkpoint loaded for generation.")


In [ ]:
# --- Loss curves ---
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(iter_hist, train_losses, label='Train loss')
    plt.plot(iter_hist, val_losses,   label='Val loss')
    plt.xlabel('Iteration')
    plt.ylabel('Cross-entropy loss')
    plt.title('Training Progress — Lyrics nano-GPT')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## 4. Generating Lyrics

This is the fun part!  A few things worth experimenting with:

- **`temperature`**: Smaller should give tight, focused output and larger will get stranger and more creative.
- **`top_k`**: Keeps the sampling from going completely off the rails.  Lower values produce more "conventional" output.
- **Opening line**: This steers the mood and rhyme scheme more than you might expect.  Try lines that end on a strong vowel sound if you want the model to rhyme back.


In [ ]:
# --- 4. Generate Lyrics ---

def generate_lyrics(prompt, max_new_tokens=400, temperature=0.8, top_k=30):
    """Generate lyrics from a starting prompt string."""
    model.eval()
    ids    = encode(prompt)
    x      = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    output = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(output[0].tolist())

# --- Try different opening lines ---
prompts = [
    "[Verse 1]\ni walked out into the rain\n",
    "[Chorus]\ncarry me home\n",
    "[Verse 1]\nshe said she'd never leave\n",
]

for prompt in prompts:
    print("=" * 60)
    print(f"PROMPT: {repr(prompt)}")
    print("=" * 60)
    print(generate_lyrics(prompt))
    print()


In [ ]:
# --- Experiment: temperature sweep ---
# See how the same prompt sounds at different temperatures
test_prompt = "[Verse 1]\ni walked out into the rain\n"

for temp in [0.6, 0.8, 1.0, 1.2]:
    print(f"--- Temperature = {temp} ---")
    print(generate_lyrics(test_prompt, max_new_tokens=200, temperature=temp, top_k=30))
    print()


## Further Exploration

You've got a lyrics generator — here are some directions to take it next:

**1. Genre specialization**
Train on a single genre or artist.  The more consistent the style, the better the output.

**2. Structural prompting**
Since we kept `[Verse]` and `[Chorus]` tags in the data, the model learns what's supposed to come after them.  Try prompting with just a tag and see if it generates the right kind of content.

**3. Rhyme analysis**
Write a small function that checks whether the generated lines rhyme at the end.  Use it to score samples and keep only the ones that rhyme.  This is a cheap post-processing step that can make output look much more convincing.
